# Assignment: A* Search on a Grid

## Objective
In this assignment you will implement and analyze the **A\* Search algorithm** on a grid-based pathfinding problem.

The goal is to understand how **heuristics guide search** and how they affect the number of explored nodes.

---

## Problem Description

You are given a **10 × 10 grid world** representing a map. Each cell in the grid can contain:

| Symbol | Meaning |
|------|------|
| `S` | Start position |
| `G` | Goal position |
| `.` | Free cell (traversable) |
| `#` | Obstacle (blocked cell) |

The agent can move in **four directions only**:

- Up
- Down
- Left
- Right

Each movement has a **uniform cost of 1**.

Your task is to compute the **shortest path from the Start (S) to the Goal (G)**.

---

## Algorithms to Compare

You will compare two algorithms:

### 1. Dijkstra’s Algorithm
Dijkstra’s algorithm explores nodes based purely on the **current path cost**:

\[
f(n) = g(n)
\]

Where:

- \(g(n)\) = cost from start to node \(n\)

It guarantees the shortest path but may explore many unnecessary nodes.

---

### 2. A\* Search Algorithm
A\* improves efficiency by adding a **heuristic estimate** of the remaining distance:

\[
f(n) = g(n) + h(n)
\]

Where:

- \(g(n)\) = cost from start to node \(n\)
- \(h(n)\) = heuristic estimate from node \(n\) to the goal

In this assignment we use the **Manhattan Distance heuristic**:

\[
h(n) = |x_n - x_g| + |y_n - y_g|
\]

This heuristic is **admissible** and **consistent** for grid movement without diagonals.

---

## Expected Output
An expected output for the given grid is shown in the Cell Output below the skeleton code.

In [73]:
grid = [
['S','.','.','.','#','.','.','.','.','.'],
['.','#','#','.','#','.','#','#','#','.'],
['.','.','.','.','.','.','.','.','#','.'],
['#','#','.','#','#','#','.','#','#','.'],
['.','.','.','.','.','#','.','.','.','.'],
['.','#','#','#','.','#','#','#','.','#'],
['.','.','.','#','.','.','.','#','.','.'],
['#','#','.','#','#','#','.','#','#','.'],
['.','.','.','.','.','.','.','.','#','.'],
['.','#','#','#','#','#','.','.','.','G']
]

rows = len(grid)
cols = len(grid[0])

In [74]:
import heapq

def neighbors(current_position):
    current_row, current_col = current_position
    neighbors_list = []
    for row, col in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        neighbor_row = current_row + row
        neighbor_col = current_col + col
        if 0 <= neighbor_row < rows and 0 <= neighbor_col < cols and grid[neighbor_row][neighbor_col] != '#':
            neighbors_list.append((neighbor_row, neighbor_col))   
    return neighbors_list

def path(parent, start, goal):
    shortest_path = []
    current_node = goal
    while current_node != start:
        shortest_path.append(current_node)
        current_node = parent[current_node]
    shortest_path.append(start)
    shortest_path.reverse()
    return shortest_path


def djk(start, goal):
    priority_queue = []
    heapq.heappush(priority_queue, (0, start))
    
    cost_from_start = {start: 0}
    parent = {start: None}
    explored = 0

    while len(priority_queue) > 0:
        cost, node = heapq.heappop(priority_queue)
        explored += 1

        if node == goal:
            final_path = path(parent, start, goal)
            return final_path, cost, explored

        if cost > cost_from_start.get(node, float('inf')):
            continue

        for n in neighbors(node):
            new_cost = cost + 1  
            if new_cost < cost_from_start.get(n, float('inf')):
                cost_from_start[n] = new_cost
                parent[n] = node
                heapq.heappush(priority_queue, (new_cost, n))

    return None, float('inf'), explored


def astar(start, goal):
    heuristic = abs(start[0] - goal[0]) + abs(start[1] - goal[1])
    priority_queue = []
    heapq.heappush(priority_queue, (heuristic, start))
    
    cost_from_start = {start: 0}
    parent = {start: None}
    explored = 0

    while len(priority_queue) > 0:
        cost, node = heapq.heappop(priority_queue)
        explored += 1

        if node == goal:
            final_path = path(parent, start, goal)
            return final_path, cost_from_start[goal], explored
        current_heuristic = abs(node[0] - goal[0]) + abs(node[1] - goal[1])

        if cost_from_start.get(node, float('inf')) < cost - current_heuristic:
            continue

        for n in neighbors(node):
            new_cost = cost_from_start[node] + 1
            if new_cost < cost_from_start.get(n, float('inf')):
                cost_from_start[n] = new_cost
                parent[n] = node
                neighbor_heuristic = abs(n[0] - goal[0]) + abs(n[1] - goal[1])
                new_estimated_total_cost = new_cost + neighbor_heuristic 
                heapq.heappush(priority_queue, (new_estimated_total_cost, n))

    return None, float('inf'), explored

In [75]:
start = goal = None

for r in range(rows):
    for c in range(cols):
        if grid[r][c] == 'S':
            start = (r, c)
        elif grid[r][c] == 'G':
            goal = (r, c)

dijkstra_path, dijkstra_cost, dijkstra_explore = djk(start, goal)
astar_path, astar_cost, astar_explore = astar(start, goal)

print("DIJKSTRA RESULT")
print(f"Path: {dijkstra_path}")
print(f"Cost: {dijkstra_cost}")
print(f"Nodes explored: {dijkstra_explore}")

print()
print()
print()

print("A* RESULT")
print(f"Path: {astar_path}")
print(f"Cost: {astar_cost}")
print(f"Nodes explored: {astar_explore}")

DIJKSTRA RESULT
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 4), (2, 5), (2, 6), (3, 6), (4, 6), (4, 7), (4, 8), (5, 8), (6, 8), (6, 9), (7, 9), (8, 9), (9, 9)]
Cost: 18
Nodes explored: 62



A* RESULT
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 4), (2, 5), (2, 6), (3, 6), (4, 6), (4, 7), (4, 8), (5, 8), (6, 8), (6, 9), (7, 9), (8, 9), (9, 9)]
Cost: 18
Nodes explored: 39
